In [0]:
# Ingestion
BASE = "/Volumes/alpenia_dev/bronze/landing"

# --- CSV Files ---
df_contracts = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ";") \
    .load(f"{BASE}/contracts/")

df_clients = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(f"{BASE}/clients/")

df_pricing = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(f"{BASE}/pricing/")

# --- Parquet Files ---

df_claims = spark.read \
    .format("parquet") \
    .load(f"{BASE}/claims/")

# --- JSON Files ---
df_web_events = spark.read \
    .format("json") \
    .load(f"{BASE}/web_events/")

df_expertises = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(f"{BASE}/expertises/")

# Optional: Verify the first few rows of each
display(df_contracts.limit(5))
display(df_clients.limit(5))
display(df_pricing.limit(5))
display(df_claims.limit(5))
display(df_web_events.limit(5))
display(df_expertises.limit(5))


In [0]:
# Contrôle des lectures : un lecteur mal paramétré ne lève pas toujours d'erreur
# (mauvais séparateur -> une seule colonne ; JSON multi-lignes sans multiLine -> _corrupt_record)
sources = {
    "contracts": df_contracts, "clients": df_clients, "pricing": df_pricing,
    "claims": df_claims, "web_events": df_web_events, "expertises": df_expertises,
}
for name, df in sources.items():
    print(f"== {name} : {df.count()} lignes, {len(df.columns)} colonnes")
    df.printSchema()
    assert "_corrupt_record" not in df.columns, f"{name} : lecture JSON incorrecte (_corrupt_record)"
    assert len(df.columns) > 1, f"{name} : une seule colonne, séparateur probablement incorrect"

In [0]:
# Ecrire les tables
# --- Write CSV Data to Bronze Tables ---

# 1. Contracts
df_contracts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_contracts")

# 2. Clients
df_clients.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_clients")

# 3. Pricing
df_pricing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_pricing")

# --- Write Parquet/JSON Data to Bronze Tables ---

# 4. Claims
df_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_claims")

# 5. Web Events
df_web_events.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_web_events")

# 6. Expertises
df_expertises.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_expertises")



In [0]:
%sql
-- Contrôle d'idempotence : nombre de lignes attendu (Partie 2 de l'énoncé, en dur) vs obtenu
SELECT 'brz_contracts'  AS table_name, count(*) AS actual, 4000  AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_contracts
UNION ALL
SELECT 'brz_clients'    AS table_name, count(*) AS actual, 3000  AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_clients
UNION ALL
SELECT 'brz_pricing'    AS table_name, count(*) AS actual, 6     AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_pricing
UNION ALL
SELECT 'brz_claims'     AS table_name, count(*) AS actual, 24000 AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_claims
UNION ALL
SELECT 'brz_web_events' AS table_name, count(*) AS actual, 90000 AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_web_events
UNION ALL
SELECT 'brz_expertises' AS table_name, count(*) AS actual, 450   AS expected, IF(actual = expected, 'OK', 'KO') AS status FROM alpenia_dev.bronze.brz_expertises;